# Using Tabular Foundation Model for Regression
### Model used: AutoGluon's **Mitra** tabular foundation model (https://huggingface.co/autogluon/mitra-regressor)
### Use case: Predicting disease progression — specifically, a quantitative measure of how much a patient's diabetes advanced one year after baseline.

### Step 0 — Install
Install AutoGluon with the Mitra extra (commented out; run once if not already installed).

In [ ]:
#!pip install autogluon.tabular[mitra]   

### Step 1 — Imports
Bring in pandas, AutoGluon's `TabularDataset`/`TabularPredictor`, the train/test splitter, and the diabetes loader.

In [ ]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error

### Step 2 — Load the data
Fetch the diabetes dataset and build a DataFrame, adding the continuous `target` column (disease progression) we want to predict.

In [ ]:
diabetes_data = load_diabetes()
diabetes_df = pd.DataFrame(diabetes_data.data, columns=diabetes_data.feature_names)
diabetes_df['target'] = diabetes_data.target   # disease progression 

print(f"Diabetes: {diabetes_df.shape}") 
diabetes_df.head()

### Step 3 — Split Datasets
Create an 80/20 train/test split

In [ ]:
diabetes_train, diabetes_test = train_test_split(diabetes_df, test_size=0.2, random_state=42)
print(diabetes_train.shape, diabetes_test.shape)

### Step 4 — Save splits to CSV
Persist the train and test sets to disk for reuse / reproducibility.

In [ ]:
diabetes_train.to_csv('diabetes_train.csv', index=False)
diabetes_test.to_csv('diabetes_test.csv', index=False)

### Step 5 — Wrap in TabularDataset
Convert the DataFrames into AutoGluon `TabularDataset` objects (a thin pandas wrapper AutoGluon expects).

In [ ]:
diabetes_train_data = TabularDataset(diabetes_train)
diabetes_test_data = TabularDataset(diabetes_test)

### Step 6 — Create the predictor
Define a regression `TabularPredictor` for the `target` column and choose where the model is saved.

In [ ]:
mitra_reg_predictor = TabularPredictor(
    label='target',
    path='./mitra_regressor_model',
    problem_type='regression'
)

### Step 7 — Train Mitra
Fit the Mitra foundation model (in-context, no fine-tuning) on up to 10,000 sampled rows.

In [ ]:
mitra_reg_predictor.fit(
    diabetes_train_data, 
    hyperparameters={
        'MITRA': {'fine_tune': False}
    },
)

### Step 8 — Predict
Generate predictions on the test set. Inference has **no** row limit — the result is a pandas Series aligned to the test index.

In [ ]:
predictions = mitra_reg_predictor.predict(diabetes_test_data)

### Step 9 — View predictions
Display the predicted values.

In [ ]:
predictions

### Step 10 — Measure error (MSE & RMSE)
Compare predictions against the true targets to compute Mean Squared Error (MSE).

In [ ]:
y_true = diabetes_test_data['target']
y_pred = predictions

# Compute MSE
mse = mean_squared_error(y_true, y_pred)
print(f"MSE: {mse:.4f}")

rmse = mse ** 0.5
print(f"RMSE: {rmse:.4f}")